# Notebook 1. The model and the control run

**Predicting Flash Floods in Working Landscapes**, Module 3, Event based evaluation and forecast skill
CUAHSI Water Science Exchange 2026, University of Missouri, Saturday November 14, 2026
Humberto Vergara and Mohamed Abdelkader, University of Iowa, IIHR Hydroscience and Engineering

## Learning objectives of Module 3

By the end of this module you will be able to

1. Explain why a single model run hides forecast uncertainty, and how an ensemble represents it.
2. Turn an ensemble of simulations into a probability of exceeding a flood threshold.
3. Verify event forecasts with the contingency table, categorical scores and the ROC curve.
4. Verify probabilistic forecasts with the Brier score, the reliability diagram, the rank histogram and the CRPS, and use the diagnosis to improve the ensemble.

This notebook covers objective 1. Notebook 2 covers objective 2, Notebook 3 objective 3 and Notebook 4 objective 4.

## What this notebook does

Operational flood forecasting rarely rests on one model run. Rainfall is uncertain, parameters are uncertain, and the model is a simplification. Forecast centres answer with an ensemble, many plausible versions of the same forecast, and issue a probability. Before we can judge such a forecast we need a model and a clear experiment. Here we meet HyMOD, a small conceptual rainfall runoff model, drive it with 40 years of daily data from the Leaf River basin, calibrate one control run, and show why that single run is not enough.

| Step | What we do |
|---|---|
| 1 | Load the catchment data and look at it |
| 2 | Read the model, one function at a time |
| 3 | Run it with textbook parameters and score it |
| 4 | Calibrate a control run with a random search |
| 5 | Show how parameters and rainfall move the forecast peak |
| 6 | Your turn |

How to run. Click a cell and press Shift + Enter. Run the cells from top to bottom.

## 0. Setup

Run this cell first. In Google Colab it copies the module to the cloud machine (the repository is public, no account is needed). On your own computer, with the notebook opened from the repository folder, it only confirms the working folder. Nothing needs to be installed, the module uses numpy and matplotlib only.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-3-evaluation"

# Step 1. If the helper files are not next to this notebook, we are in Colab. Copy the branch.
if not Path("hymod_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")

# Step 2. Make the helper files importable.
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt

import hymod_tools as h             # the model and the scores (open hymod_tools.py to read it)
import nb_style                     # one plotting style for the four notebooks
from nb_style import PALETTE
nb_style.set_style()

print("Calibrated control parameters shipped with the module:")
for k, v in h.CONTROL_PARS.items():
    print(f"   {k:>4} = {v}")

## 1. The catchment and the data

The Leaf River basin in Mississippi (about 1,944 square kilometres) is a classic test catchment. The file `LeafCatch.dat` holds 40 years of daily data in three columns, all as catchment averages in millimetres per day.

| Column | Name | Meaning |
|---|---|---|
| 1 | `PP` | Precipitation, the model input |
| 2 | `PET` | Potential evapotranspiration, the atmospheric demand |
| 3 | `QQ` | Observed streamflow, the truth we verify against |

Working in millimetres per day puts rain, evaporation and streamflow on the same axis, so the water balance stays visible.

In [ ]:
# Step 1. Read the three columns.
PP, PET, QQ = h.load_leaf_river("LeafCatch.dat")
n_total = len(PP)
print(f"Record length: {n_total} days, about {n_total / 365:.0f} years")

In [ ]:
# Step 2. Summary statistics, one line per variable.
print(f"{'Variable':<28}{'mean':>8}{'median':>9}{'max':>9}")
for name, arr in [("Precipitation PP (mm/day)", PP),
                  ("Potential ET PET (mm/day)", PET),
                  ("Observed flow QQ (mm/day)", QQ)]:
    print(f"{name:<28}{arr.mean():>8.2f}{np.median(arr):>9.2f}{arr.max():>9.2f}")

# Step 3. Long term water balance. The runoff ratio is streamflow divided by rainfall.
print(f"\nRunoff ratio QQ/PP = {QQ.mean() / PP.mean():.2f}, "
      f"about {100 * QQ.mean() / PP.mean():.0f} percent of the rain leaves as streamflow")

The first two years show the usual structure of a humid catchment. Rain comes in pulses, evaporation follows the seasons, and streamflow answers each storm with a sharp rise and a long recession. The model has to reproduce that lag and smoothing.

In [ ]:
days = np.arange(2 * 365)                       # the first two years

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

# Rain drawn downward, the meteorological convention.
axes[0].bar(days, PP[:len(days)], width=1.0, color=PALETTE["control"])
axes[0].invert_yaxis()
axes[0].set_ylabel("PP\n(mm/day)")
axes[0].set_title("Leaf River, first two years of daily data")

# Potential evapotranspiration, a smooth seasonal curve.
axes[1].plot(days, PET[:len(days)], color=PALETTE["accent"])
axes[1].set_ylabel("PET\n(mm/day)")

# Observed streamflow, peaks and recessions.
axes[2].fill_between(days, QQ[:len(days)], color=PALETTE["control"], alpha=0.35)
axes[2].plot(days, QQ[:len(days)], color=PALETTE["obs"], lw=1.0)
axes[2].set_ylabel("QQ\n(mm/day)"); axes[2].set_xlabel("day of record")
plt.tight_layout(); plt.show()

## 2. The model, one function at a time

HyMOD is a conceptual model. It routes water through a few storage tanks governed by six parameters. Water passes three stages.

1. Soil moisture accounting. A probability distributed store decides how much of the rain becomes excess and how much is stored or evaporated.
2. Quick and slow split. A fraction `Alp` of the excess takes the fast path, the rest the slow path.
3. Routing. The fast path crosses a cascade of `Nq` leaky tanks, the slow path one slow tank. The two outflows add up to streamflow.

| Parameter | Controls | Range |
|---|---|---|
| `Huz` | Maximum height of the soil store | 0 to about 500 mm |
| `B` | Spread of storage capacities in the catchment | 0 to 2 |
| `Alp` | Quick against slow split | 0 to 1 |
| `Nq` | Number of quick tanks | integer, 1 or more |
| `Kq` | Release rate of the quick tanks | 0 to 1 per day |
| `Ks` | Release rate of the slow tank | 0 to 1 per day |

In [ ]:
# A schematic of the water path. Rain in, soil store, split, routing, streamflow out.
fig, ax = plt.subplots(figsize=(11, 4.6)); ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, 5)

def box(x, y, w, hgt, text, fc):
    ax.add_patch(plt.Rectangle((x, y), w, hgt, fc=fc, ec="k", lw=1.4, alpha=0.9))
    ax.text(x + w / 2, y + hgt / 2, text, ha="center", va="center", fontsize=10, fontweight="bold")

def arrow(x0, y0, x1, y1, text="", color="k"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=1.8, color=color))
    if text:
        ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.18, text, ha="center", fontsize=9, color=color)

arrow(0.6, 4.6, 1.9, 3.7, "rain PP", PALETTE["control"])
arrow(1.9, 3.9, 0.7, 4.7, "ET", PALETTE["accent"])
box(1.9, 2.9, 2.0, 1.2, "Soil store\nHuz, B", "#cfe3f5")
arrow(3.9, 3.5, 5.0, 4.1, "quick, Alp x OV", PALETTE["ens_mean"])
arrow(3.9, 3.5, 5.0, 2.0, "slow, (1 - Alp) x OV", PALETTE["control"])
box(5.0, 3.7, 2.4, 0.9, "Nash cascade\nNq tanks, Kq", "#f6d9c2")
box(5.0, 1.6, 2.4, 0.9, "Slow tank\nKs", "#cfe3f5")
arrow(7.4, 4.1, 8.5, 3.1, "Qq", PALETTE["ens_mean"])
arrow(7.4, 2.0, 8.5, 2.9, "Qs", PALETTE["control"])
box(8.5, 2.5, 1.2, 0.9, "Q = Qq + Qs", "#d6f0e2")
ax.text(5.0, 0.6, "OV is the precipitation excess leaving the soil store each day", fontsize=9.5, ha="left")
plt.show()

### 2.1 The soil store

A catchment is a patchwork of places that saturate at different rainfall amounts. The probability distributed store captures this with one parameter, `B`, that sets the spread of storage capacities. The code is short. Read the comments, each line maps to a process.

In [ ]:
print(inspect.getsource(h.pdm_step))

### 2.2 The routing

A linear reservoir releases water in proportion to what it holds, outflow equals `K` times storage. A chain of `N` such tanks, the Nash cascade, turns a sharp pulse into the smooth lagged shape of a hydrograph.

In [ ]:
print(inspect.getsource(h.nash_step))

### 2.3 The full model

`hymod01` marches through time. Each day it calls the soil store, splits the excess, routes the quick and slow parts, and adds them. The tank states at the end of a day start the next day. That memory produces the recessions.

In [ ]:
print(inspect.getsource(h.hymod01))

## 3. A first run with textbook parameters

The original tutorial ships a default parameter set. We run it over the first five years, the evaluation window used in all four notebooks, and score it with three deterministic measures.

- RMSE, root mean square error in mm per day, lower is better.
- NSE, Nash Sutcliffe efficiency. 1 is perfect, 0 means no better than the mean flow, negative means worse.
- KGE, Kling Gupta efficiency, which balances correlation, bias and variability.

In [ ]:
# Step 1. The evaluation window and the observed flow in it.
N_EVAL = 365 * 5
Qobs = QQ[:N_EVAL]

# Step 2. Run the model with the default parameters.
base = h.hymod01(PP[:N_EVAL], PET[:N_EVAL], h.DEFAULT_PARS)
Qbase = base["Q"]
print("Default parameters:", h.DEFAULT_PARS)

In [ ]:
# Step 3. Score it.
print(f"RMSE = {h.rmse(Qbase, Qobs):6.3f} mm/day")
print(f"NSE  = {h.nse(Qbase, Qobs):6.3f}")
print(f"KGE  = {h.kge(Qbase, Qobs):6.3f}")
print(f"simulated peak {Qbase.max():.2f} mm/day against observed peak {Qobs.max():.2f} mm/day")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(Qobs, color=PALETTE["obs"], lw=1.2, label="observed")
ax.plot(Qbase, color=PALETTE["control"], lw=1.4, label="HyMOD, default parameters")
ax.set_xlabel("day"); ax.set_ylabel("streamflow (mm/day)")
ax.set_title("Default parameters, the peaks are badly under predicted")
ax.legend(); plt.tight_layout(); plt.show()

What happened. The default `Kq` of 0.02 releases two percent of the quick tanks per day, far too slow for quick flow. The model hoards water, never builds a peak, and the NSE is negative. A model is only as good as its parameters, and we rarely know them exactly.

## 4. Calibrating a control run

Before issuing forecasts, a centre calibrates the model against past observations. We use the simplest search. Draw random parameter sets, run each one, keep the best NSE. Real systems use smarter optimisers, the idea is the same.

In [ ]:
rng = np.random.default_rng(42)         # fixed seed, the search is reproducible
N_SAMPLES = 400
best_score, best_pars = -np.inf, None

# Step 1. Random search. Each trial is one parameter set and one model run.
for _ in range(N_SAMPLES):
    trial = dict(Huz=rng.uniform(10, 500), B=rng.uniform(0, 2), Alp=rng.uniform(0, 1),
                 Nq=int(rng.integers(1, 6)), Kq=rng.uniform(0.05, 0.95), Ks=rng.uniform(0.001, 0.1))
    score = h.nse(h.hymod01(PP[:N_EVAL], PET[:N_EVAL], trial)["Q"], Qobs)
    if score > best_score:                 # keep the best set seen so far
        best_score, best_pars = score, trial

print(f"best of {N_SAMPLES} random sets: NSE = {best_score:.3f}")
for k, v in best_pars.items():
    print(f"   {k:>4} = {v:.4f}" if isinstance(v, float) else f"   {k:>4} = {v}")

In [ ]:
# Step 2. The same set is stored as h.CONTROL_PARS and reused by Notebooks 2 to 4.
control = h.hymod01(PP[:N_EVAL], PET[:N_EVAL], h.CONTROL_PARS)
Qctrl = control["Q"]
print(f"control run: NSE = {h.nse(Qctrl, Qobs):.3f}, RMSE = {h.rmse(Qctrl, Qobs):.3f} mm/day, KGE = {h.kge(Qctrl, Qobs):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(Qobs, color=PALETTE["obs"], lw=1.2, label="observed")
ax.plot(Qctrl, color=PALETTE["ens_mean"], lw=1.4, label=f"calibrated control, NSE = {h.nse(Qctrl, Qobs):.2f}")
ax.set_xlabel("day"); ax.set_ylabel("streamflow (mm/day)")
ax.set_title("The calibrated control run follows the timing and size of the peaks")
ax.legend(); plt.tight_layout(); plt.show()

Two diagnostic views. The scatter shows where the model errs, the points off the one to one line. The flow duration curve shows whether the model reproduces the whole range of flows, from the frequent low flows on the right to the rare high flows on the left that matter for floods.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

# Simulated against observed with the one to one line.
axes[0].scatter(Qobs, Qctrl, s=10, color=PALETTE["control"], alpha=0.4)
lim = [0, max(Qobs.max(), Qctrl.max()) * 1.05]
axes[0].plot(lim, lim, "k--", lw=1)
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("observed Q (mm/day)"); axes[0].set_ylabel("simulated Q (mm/day)")
axes[0].set_title("Simulated against observed")

# Flow duration curves on a log axis.
ex_o, fd_o = h.flow_duration_curve(Qobs)
ex_c, fd_c = h.flow_duration_curve(Qctrl)
axes[1].semilogy(ex_o, fd_o, color=PALETTE["obs"], label="observed")
axes[1].semilogy(ex_c, fd_c, color=PALETTE["ens_mean"], label="control")
axes[1].set_xlabel("exceedance probability (percent)"); axes[1].set_ylabel("Q (mm/day, log scale)")
axes[1].set_title("Flow duration curve"); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Why one forecast is not enough

The control run depends on the parameters we happened to pick and on rainfall assumed exactly right. Both are uncertain. Two sweeps make that visible on one flood event.

In [ ]:
# Step 1. Zoom on one flood event.
event = slice(450, 560)
xs = np.arange(event.start, event.stop)

# Step 2. Sweep the quick flow rate Kq, everything else at the control values.
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(xs, Qobs[event], color=PALETTE["obs"], lw=2, label="observed", zorder=5)
for kq in np.linspace(0.2, 0.9, 8):
    pars = dict(h.CONTROL_PARS); pars["Kq"] = kq
    Q = h.hymod01(PP[:N_EVAL], PET[:N_EVAL], pars)["Q"]
    ax.plot(xs, Q[event], color=PALETTE["ensemble"], lw=1.1)
ax.plot([], [], color=PALETTE["ensemble"], label="HyMOD with eight values of Kq")
ax.set_xlabel("day"); ax.set_ylabel("streamflow (mm/day)")
ax.set_title("One parameter, Kq, moves the peak over a wide range")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Step 3. Keep the control parameters, scale the rainfall from 60 to 140 percent.
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(xs, Qobs[event], color=PALETTE["obs"], lw=2, label="observed", zorder=5)
for mult in np.linspace(0.6, 1.4, 9):
    Q = h.hymod01(PP[:N_EVAL] * mult, PET[:N_EVAL], h.CONTROL_PARS)["Q"]
    ax.plot(xs, Q[event], color=PALETTE["skill"], lw=1.1, alpha=0.8)
ax.plot([], [], color=PALETTE["skill"], label="HyMOD with scaled rainfall")
ax.set_xlabel("day"); ax.set_ylabel("streamflow (mm/day)")
ax.set_title("Rainfall depth error goes straight into the peak")
ax.legend(); plt.tight_layout(); plt.show()

Rainfall error is usually the largest source of streamflow forecast error, which is why Module 1 spent two hours on it.

## 6. Your turn

Sweep a second parameter, the split `Alp`, over the same event and report how far the simulated peak moves. Fill the block below.

In [ ]:
#################### Your Code Goes Here #########################
# Step 1. Eight values of Alp between 0.2 and 0.9 (np.linspace).


# Step 2. For each value, copy h.CONTROL_PARS, replace "Alp", run the model
#         over the evaluation window, and store the peak of Q inside the event slice.
peaks = []


# Step 3. Print the smallest and largest simulated peak, and the observed peak of the event.

###################################################################

Questions for the group

1. Which parameter moved the peak more, `Kq` or `Alp`? Why would that be, given what each one controls?
2. If you could measure one thing better, the rainfall or a parameter, which one would reduce the spread the most?

## Recap

- HyMOD turns rain into streamflow through one soil store and two routing paths, with six parameters.
- The default parameters gave a negative NSE. A random search found a credible control run, stored as `h.CONTROL_PARS`.
- One parameter or the rainfall depth alone swings the forecast peak widely. A single run hides that.

Notebook 2 draws many plausible parameter and rainfall combinations, runs them all, and turns the collection into a probability.

## References and credits

- Boyle, D. P., 2001. Multicriteria calibration of hydrologic models. PhD thesis, University of Arizona. The origin of HyMOD.
- Gupta, H. V., 2005. HyMOD tutorial code, HWR 521, University of Arizona. The MATLAB source ported in `hymod_tools.py`.
- Moore, R. J., 2007. The PDM rainfall runoff model. Hydrology and Earth System Sciences, 11, 483 to 499.
- Nash, J. E. and J. V. Sutcliffe, 1970. River flow forecasting through conceptual models, part I. Journal of Hydrology, 10, 282 to 290.
- Gupta, H. V., H. Kling, K. K. Yilmaz and G. F. Martinez, 2009. Decomposition of the mean squared error and NSE performance criteria. Journal of Hydrology, 377, 80 to 91.
- Adapted from the Ensemble Forecast Evaluation series prepared by Humberto Vergara and Mohamed Abdelkader for the CIROH Summer Institute 2026.